#### A typical RAG system searches uploaded documents to find a relevant snippet. Agentic RAG, instead of always looking up the information, stops to consider whether it really needs to search or if it can answer on its own.

##### Agentic RAG pipeline using Python, LangChain, and a lightweight Google model.
- LangChain to manage the process
- ChromaDB for storing vectors 
- Google’s Flan-T5 as the local language model

In [1]:
import os
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

/root/opt/la-i-b/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_2299121/1623642887.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
# Testing the available for today models, cause we need a free tier model
client = InferenceClient(token=os.getenv("HF_TOKEN"))

models_to_test = [
      # Qwen family
      "Qwen/Qwen2.5-7B-Instruct",
      "Qwen/Qwen2.5-72B-Instruct",
      
      # Meta Llama family
      "meta-llama/Meta-Llama-3-8B-Instruct",
      "meta-llama/Meta-Llama-3.1-8B-Instruct",
      "meta-llama/Meta-Llama-3.1-70B-Instruct",
      
      # Google Gemma family
      "google/gemma-2-9b-it",
      "google/gemma-2-27b-it",
      
      # Mistral family
      "mistralai/Mistral-7B-Instruct-v0.3",
      "mistralai/Mixtral-8x7B-Instruct-v0.1",
      "mistralai/Mistral-Small-24B-Instruct-2501",
      
      # Microsoft Phi family
      "microsoft/Phi-3-mini-4k-instruct",
      "microsoft/Phi-4-mini-instruct",
      
      # DeepSeek
      "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
      
      # Others
      "HuggingFaceH4/zephyr-7b-beta",
      "NousResearch/Hermes-3-Llama-3.1-8B",
]

for m in models_to_test:
    try:
        result = client.chat_completion(
            messages=[{"role": "user", "content": "Say hello"}],
            model=m,
            max_tokens=10
        )
        print(f"{m}: WORKS - {result.choices[0].message.content}")
    except Exception as e:
        print(f"{m}: FAILED - {str(e)[:80]}")
        

Qwen/Qwen2.5-7B-Instruct: FAILED - (Request ID: Root=1-6a61e31f-551600fc287390e86bc5bb3f;663ae1ee-462c-46ec-b8ed-d2
Qwen/Qwen2.5-72B-Instruct: WORKS - Hello! How can I help you today?
meta-llama/Meta-Llama-3-8B-Instruct: FAILED - (Request ID: Root=1-6a61e320-1157a0fd39c4ef9414d7bb65;0f633887-a47b-475b-8d24-60
meta-llama/Meta-Llama-3.1-8B-Instruct: FAILED - (Request ID: Root=1-6a61e320-151ea4340f45600247642469;70b2ad32-5fa9-4a15-86bc-fd
meta-llama/Meta-Llama-3.1-70B-Instruct: FAILED - (Request ID: Root=1-6a61e320-1cfda4e54bd433f12977d533;3501fe74-bd55-4e7a-be45-20
google/gemma-2-9b-it: FAILED - (Request ID: Root=1-6a61e320-4e1331f04b91de9c5ddf1209;f2efd6c4-8363-43af-bc16-3a
google/gemma-2-27b-it: FAILED - (Request ID: Root=1-6a61e320-7dbd2c747e6af4b047368e15;13688bb6-78bb-4179-8188-21
mistralai/Mistral-7B-Instruct-v0.3: FAILED - (Request ID: Root=1-6a61e321-0eb5390370eef2d268d19c67;887f9d93-748e-46a8-a27e-30
mistralai/Mixtral-8x7B-Instruct-v0.1: FAILED - (Request ID: Root=1-6a61e321-26c

In [ ]:
# Setting up the model and the inference client

client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "Qwen/Qwen2.5-72B-Instruct"

def ask_llm(prompt, max_tokens=200):
    result = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=max_tokens
    )
    return result.choices[0].message.content.strip()

In [3]:
def load_docs(folder_path):
    """Loads all PDF files from a folder and returns a list of documents."""
    docs = []
    # loading pdf files one page at a time
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            docs.extend(loader.load())
    return docs

docs = load_docs("../data/raw/pdf")
print("PDF Pages Loaded:", len(docs))

# Chunking / spilitting the documents into smaller pieces

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # LLMs can only read a certain amount of text at once - the context window
    chunk_overlap=80    # Instead of just cutting the text, we let the chunks overlap a bit. Sentences aren’t split in half at the edge of a chunk, so the meaning (or semantic context) is preserved across breaks.
)
chunks = text_splitter.split_documents(docs)
print("Chunks Created:", len(chunks))

PDF Pages Loaded: 3
Chunks Created: 13


In [4]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-25T11:24:39+00:00', 'source': '../data/raw/pdf/Full-49.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='3.6.2 https://phys.libretexts.org/@go/page/64095\nCC BY-SA 3. 0 | Image courtesy of Wikimedia Author: Lutfar Rahman Nirjhar.\nRight at totality and during totality, you will be able to see several things associated with totality.\nThe Diamond ring looks just like its name. It is the last bright sli ver of the Sun visible as the Moon covers the Sun, or right after\nthe end of totality as the Sun is being uncovered by the Moon.'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-25T11:24:39+00:00', 'source': '../data/raw/pdf/Full-49.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='the end of totality as the Sun is being uncovered by the Moon.\nBaily’s Beads are spots of light that appear as

In [5]:
# Embeddings and Vector Store - convert text into numbers (vectors) and sore it in chroma vector database for retrieval
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # ll-MiniLM-L6-v2 - a small, fast model 

# Save texts into Chroma vector DB
texts = [c.page_content for c in chunks]
db = Chroma(
    collection_name="rag_store",
    embedding_function=embedding_model
)
db.add_texts(texts)

# Retriever
retriever = db.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2680.86it/s]


In [6]:
# Agent Controller

def agent_controller(query):
    """Instead of sending everything to the database, this controller analyzes the user’s intent:
        - Does the user want data from the file? Action: Search
        - Is the user just chatting or asking for general knowledge? Action: Direct"""
    q = query.lower()
    if any(word in q for word in ["pdf", "document", "data", "summarize",
                                   "information", "find", "what does",
                                   "according to", "from the"]):
        return "search"
    return "direct"

In [ ]:
# RAG : Question - Answer

def rag_answer(query):
    """Retrieval-Augmented Generation (RAG) answer function.

    Args:
        query (str): The question to be answered.

    Returns:
        str: The answer generated by the RAG system.
    """
    action = agent_controller(query)

    if action == "search":
        print(f"  Agent decided to SEARCH documents for: '{query}'")
        results = retriever.invoke(query)
        context = "\n".join([r.page_content for r in results])
        prompt = f"""Use this context to answer the question.
                    If the answer is not in the context, say "I don't have that information."

                    Context:
                    {context}

                    Question:
                    {query}

                    Answer:"""
    else:
        print(f"  Agent decided to answer DIRECTLY: '{query}'")
        prompt = query

    return ask_llm(prompt)

In [12]:
# Testing
print(rag_answer("Give me a 5-point summary from the PDF"))
print("-" * 50)
print(rag_answer("What is an Ideal Colour for painting ships? Explain in 50 words."))

  Agent decided to SEARCH documents for: 'Give me a 5-point summary from the PDF'
I don't have that information. The provided context does not contain a 5-point summary from a PDF. It includes references to lunar eclipses, a quote from the Bible (Amos 8:9), and descriptions of phenomena visible during a solar eclipse, but it does not provide a summary of a PDF.
--------------------------------------------------
  Agent decided to answer DIRECTLY: 'What is an Ideal Colour for painting ships? Explain in 50 words.'
The ideal color for painting ships is often a shade of gray or white. These colors reflect sunlight, reducing heat absorption and UV damage. They also offer good camouflage at sea and are less likely to show dirt and grime, maintaining a clean and professional appearance.
